# **Task 2 — DeiT**: does seeing more pixels help?

This week's seminar was about [DeiT](https://proceedings.mlr.press/v139/touvron21a.html): a ViT trained on ImageNet alone, with distillation, and fine-tuned at a higher resolution at the end. The paper reports that going from 224×224 to 384×384 images makes the model more accurate (Table 10). Today you check whether that holds for **our** model and **our** data.

This notebook loads the data and the model and gives you a **fixed evaluation**, so that every group measures its result the same way. Your own work starts after the baseline.

<details>
<summary>Reach out here if you have any problems</summary>
Taras Kutsyk — taras.kutsyk@doctoral.uj.edu.pl
</details>

# The task

**Question.** Does seeing more pixels help? Make the model run at 384×384, fine-tune it within the budget, and **answer the question for this model and this data, with evidence**. Then explain *why* you got the result you got, and what they teach you.

*Hint:* be wary of what you compare with, and of how your claim relates to the paper's.

**Fixed:**
- the starting model and the dataset;
- the `evaluate()` function below;
- the fine-tuning budget: **at most 100,000 training images per fine-tuned model** (about 11 passes over the training set);
- write the resizing of the position embeddings yourselves: no ready-made resizing (e.g. timm's `img_size=` argument or `resample_abs_pos_embed`).

Everything else is your choice.

**Bring to the lab:** your code + a short deck with your answer, the evidence for it, and your explanation. If your group defends this week, **every member should be able to explain any piece of the code**.

# Setup

**Step 0.** Runtime → Change runtime type → **T4 GPU**. Everything we need (PyTorch, torchvision, timm) is already installed on Colab.

**Step 1. Data.** [Imagewoof](https://github.com/fastai/imagenette): 10 dog breeds from ImageNet, 9,025 training and 3,929 validation photos, **at full size** (about 1.3 GB, under a minute to download).

In [ ]:
import os, tarfile, time, urllib.request
import torch, timm
from torchvision import datasets

device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "Switch to a GPU runtime: Runtime -> Change runtime type -> T4 GPU"

URL = "https://s3.amazonaws.com/fast-ai-imageclas/imagewoof2.tgz"
# we use the full-size image sets on purpose, not the smaller "-320" versions
# can you tell why? (in the deck)
if not os.path.exists("imagewoof2/val"):
    urllib.request.urlretrieve(URL, "imagewoof2.tgz")
    with tarfile.open("imagewoof2.tgz") as t:
        t.extractall(filter="data")
for split in ["train", "val"]:
    print(split, sum(len(f) for _, _, f in os.walk(f"imagewoof2/{split}")), "photos")

train 9025 photos
val 3929 photos


**Step 2. Model.** A distilled DeiT-S from the paper (12 blocks, 384 dims, 22M parameters, the DeiT-S⚗ of Table 5), trained on ImageNet at 224×224, together with the exact preprocessing it was trained with.

The model predicts one of 1000 ImageNet classes, so we map each Imagewoof folder (e.g. `n02086240` = Shih-Tzu) to its **ImageNet index**: the position of that breed among the model's 1000 outputs. You will need `to_imagenet` for your fine-tuning too.

<details>
<summary>Running the cell below can ask for your HF_TOKEN secret</summary>
Just press "Grant Access", it's automatically generated by Colab
</details>

In [ ]:
model = timm.create_model("deit_small_distilled_patch16_224", pretrained=True).to(device).eval()
cfg = timm.data.resolve_data_config({}, model=model)  # the preprocessing the model was trained with

# ImageFolder labels the 10 folders 0..9, but the model outputs 1000 ImageNet classes:
# to_imagenet[label] translates a folder label into the ImageNet class index
wnids = list(timm.data.ImageNetInfo().label_names())
breeds = datasets.ImageFolder("imagewoof2/val").classes
to_imagenet = torch.tensor([wnids.index(w) for w in breeds], device=device)
print(cfg)
print("folder -> ImageNet index:", dict(zip(breeds, to_imagenet.tolist())))

model.safetensors: reconstructing file:   0%|          |  0.00B / 89.8MB            

model.safetensors: downloading bytes:           |  0.00B            

{'input_size': (3, 224, 224), 'interpolation': 'bicubic', 'mean': (0.485, 0.456, 0.406), 'std': (0.229, 0.224, 0.225), 'crop_pct': 0.9, 'crop_mode': 'center'}
folder -> ImageNet index: {'n02086240': 155, 'n02087394': 159, 'n02088364': 162, 'n02089973': 167, 'n02093754': 182, 'n02096294': 193, 'n02099601': 207, 'n02105641': 229, 'n02111889': 258, 'n02115641': 273}


**Step 3. The fixed evaluation** (DON'T modify it). `evaluate(model, resolution)` resizes the validation photos for the given resolution exactly as the model's own preprocessing does, and returns the **top-1 accuracy over the 10 breeds**: the model's prediction is the highest-scoring of its 10 dog outputs.

In [ ]:
@torch.no_grad()
def evaluate(model, resolution):
    """The fixed evaluation for task 2. Do not change it; report exactly what it prints.

    model      - any model that takes (batch, 3, resolution, resolution) images and returns 1000 logits
    resolution - the input size: 224 for the original model, e.g. 384 for yours
    returns: accuracy - top-1 over the 10 breeds, on the 3,929 Imagewoof validation photos
    """
    model.eval()
    tf = timm.data.create_transform(**dict(cfg, input_size=(3, resolution, resolution)))
    loader = torch.utils.data.DataLoader(datasets.ImageFolder("imagewoof2/val", transform=tf),
                                         batch_size=64, num_workers=2)
    correct = n = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(x)
        correct += (logits[:, to_imagenet].argmax(-1) == y).sum().item()  # best of the 10 breeds
        n += len(y)
    return correct / n

**Step 4. Baseline.** Takes about half a minute. You should get **top-1 ≈ 97.20%** at 224. If not, something in the setup went wrong.

In [ ]:
print(f"baseline DeiT-S distilled @224: top-1 {evaluate(model, 224):.2%}")

baseline DeiT-S distilled @224: top-1 97.20%


# Before you start

Think about **what exactly breaks in the model when you pass it higher-resolution images?** What's the simplest, most effective way to fix that?

Training photos are in `imagewoof2/train` (same folder structure as `val`).

# Your solution

Make the model run at 384, fine-tune within the budget, and collect the evidence for your answer. Good luck!

```markdown
## Person 3: Full Fine-Tuning & Progressive Curriculum @384

To ensure no progress or artifacts are lost if the kernel restarts, we will mount Google Drive, save our training history/logs, and store our best model checkpoints there.
```

In [ ]:
import os
import copy
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, datasets

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Setup Save Directory
SAVE_DIR = "/content/drive/MyDrive/deit_seminar_task2"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"All checkpoints and logs will be saved to: {SAVE_DIR}")

Mounted at /content/drive
All checkpoints and logs will be saved to: /content/drive/MyDrive/deit_seminar_task2


```markdown
### 1. Position Embedding Interpolation Function
As per the rules, we must manually write the positional embedding interpolation (without using `timm`'s `img_size` argument or `resample_abs_pos_embed`).
We will interpolate the spatial grid portion of `pos_embed` (excluding the class token and distillation token) to the target grid size.
```

In [ ]:
import torch.nn.functional as F

def interpolate_pos_embed(model_to_modify, new_size=24, num_extra_tokens=2):
    """
    Interpolates the positional embeddings from 14x14 grid (224px) to new_size x new_size grid.
    DeiT Small Distilled has 2 extra tokens: Class token & Distillation token.
    """
    # Get existing pos_embed shape: [1, 198, 384]
    pos_embed = model_to_modify.pos_embed
    embedding_dim = pos_embed.shape[-1]

    # Split tokens and spatial grid
    extra_tokens = pos_embed[:, :num_extra_tokens]
    grid_tokens = pos_embed[:, num_extra_tokens:]

    # Reshape grid tokens to 2D grid
    # Originally 14x14 for 224x224 input resolution
    old_size = int(grid_tokens.shape[1] ** 0.5)
    grid_tokens = grid_tokens.reshape(1, old_size, old_size, embedding_dim).permute(0, 3, 1, 2)

    # Interpolate using bicubic interpolation
    grid_tokens = F.interpolate(
        grid_tokens,
        size=(new_size, new_size),
        mode='bicubic',
        align_corners=False
    )

    # Reshape back to sequence form
    grid_tokens = grid_tokens.permute(0, 2, 3, 1).reshape(1, -1, embedding_dim)

    # Concat extra tokens back
    new_pos_embed = torch.cat((extra_tokens, grid_tokens), dim=1)

    # Update model parameters
    model_to_modify.pos_embed = nn.Parameter(new_pos_embed)

    # Crucial step: update timm's internal expected image size to bypass strict shape assertion
    patch_size = model_to_modify.patch_embed.patch_size
    model_to_modify.patch_embed.img_size = (new_size * patch_size[0], new_size * patch_size[1])
    model_to_modify.patch_embed.grid_size = (new_size, new_size)

    return model_to_modify

```markdown
### 2. Dataset Setup and Helper Utilities
```

In [ ]:
def get_train_loader(resolution, batch_size=64):
    tf = transforms.Compose([
        transforms.RandomResizedCrop(resolution, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    ds = datasets.ImageFolder("imagewoof2/train", transform=tf)
    return torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)

criterion = nn.CrossEntropyLoss()

```markdown
### 3. Solution C: Full Fine-Tuning @384
We initialize the model, scale position embeddings to $24 \times 24$ ($384$ px), configure differential learning rates, and train for 11 epochs (budget limit of 99,275 training images total).
If a saved checkpoint exists, we will load it immediately to avoid unnecessary retraining.
```

In [ ]:
model_c_path = os.path.join(SAVE_DIR, "model_384_full.pth")
log_c_path = os.path.join(SAVE_DIR, "history_full.json")

# Instantiate a clean copy of the baseline model
model_384_full = copy.deepcopy(model)
model_384_full = interpolate_pos_embed(model_384_full, new_size=24)

for param in model_384_full.parameters():
    param.requires_grad = True

optimizer_c = optim.AdamW([
    {'params': [p for n, p in model_384_full.named_parameters()
                if 'head' not in n and 'pos_embed' not in n], 'lr': 1e-5},
    {'params': [p for n, p in model_384_full.named_parameters()
                if 'pos_embed' in n], 'lr': 1e-4},
    {'params': [p for n, p in model_384_full.named_parameters()
                if 'head' in n], 'lr': 1e-4},
], weight_decay=0.05)

scheduler_c = optim.lr_scheduler.CosineAnnealingLR(optimizer_c, T_max=11)
train_loader_384 = get_train_loader(384, batch_size=64)

history_c = []
budget_c_used = 0

if os.path.exists(model_c_path) and os.path.exists(log_c_path):
    print("Found saved checkpoint for Solution C. Loading parameters and logs...")
    model_384_full.load_state_dict(torch.load(model_c_path))
    with open(log_c_path, 'r') as f:
        history_c = json.load(f)
    print("Loaded validation history:", history_c)
else:
    print("Starting training for Solution C (Full Fine-Tuning @384)... ")
    for epoch in range(11):
        model_384_full.train()
        for x, y in train_loader_384:
            x, y = x.to(device), y.to(device)
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model_384_full(x)
                loss = criterion(logits[:, to_imagenet], y)

            optimizer_c.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model_384_full.parameters(), max_norm=1.0)
            optimizer_c.step()
            budget_c_used += len(x)

        scheduler_c.step()
        val_acc = evaluate(model_384_full, 384)
        history_c.append({"epoch": epoch + 1, "val_acc": val_acc})
        print(f"Epoch {epoch+1}: val {val_acc:.2%}")

    # Save checkpoint and history
    torch.save(model_384_full.state_dict(), model_c_path)
    with open(log_c_path, 'w') as f:
        json.dump(history_c, f)
    print("Solution C training completed and saved!")

Starting training for Solution C (Full Fine-Tuning @384)... 
Epoch 1: val 97.38%
Epoch 2: val 97.33%
Epoch 3: val 97.43%
Epoch 4: val 97.63%
Epoch 5: val 97.56%
Epoch 6: val 97.61%
Epoch 7: val 97.51%
Epoch 8: val 97.61%
Epoch 9: val 97.61%
Epoch 10: val 97.56%
Epoch 11: val 97.58%
Solution C training completed and saved!


```markdown
### 4. Solution E: Progressive Resolution Curriculum
Instead of jumping straight to 384, we transition through `224 -> 256 -> 320 -> 384`. To avoid positional embedding degradation (blur) from sequential interpolation, we always keep a pristine 224 model copy and interpolate directly from it to the targeted grid sizes.
```

In [ ]:
model_e_path = os.path.join(SAVE_DIR, "model_prog.pth")
log_e_path = os.path.join(SAVE_DIR, "history_prog.json")

model_prog = copy.deepcopy(model)
original_clean_model = copy.deepcopy(model)

# Schedule designed to respect the budget (<100,000 images total)
schedule = [
    (224, 14, 4, 5e-5), # 4 epochs
    (256, 16, 4, 2e-5), # 4 epochs
    (320, 20, 1, 1e-5), # 1 epoch
    (384, 24, 2, 5e-6), # 2 epochs
] # Total 11 epochs = ~99,275 images

history_e = []
budget_e_used = 0

if os.path.exists(model_e_path) and os.path.exists(log_e_path):
    print("Found saved checkpoint for Solution E. Loading...")
    model_prog = interpolate_pos_embed(model_prog, new_size=24)
    model_prog.load_state_dict(torch.load(model_e_path))
    with open(log_e_path, 'r') as f:
        history_e = json.load(f)
    print("Loaded validation history:", history_e)
else:
    print("Starting training for Solution E (Progressive Curriculum)... ")
    for resolution, grid_size, epochs, lr in schedule:
        print(f"\nTransitioning to Resolution: {resolution} (Grid: {grid_size}x{grid_size}) with LR: {lr}")

        # Interpolate cleanly from the original saved copy to prevent repeated interpolation blur
        temp_clean_model = copy.deepcopy(original_clean_model)
        temp_clean_model = interpolate_pos_embed(temp_clean_model, new_size=grid_size)

        # Copy the current updated weights over to the correctly structured temp model (excluding pos_embed)
        current_state = model_prog.state_dict()
        clean_state = temp_clean_model.state_dict()
        for k in clean_state.keys():
            if k != 'pos_embed':
                clean_state[k] = current_state[k]

        model_prog = temp_clean_model
        model_prog.load_state_dict(clean_state)

        loader = get_train_loader(resolution, batch_size=64)
        optimizer_e = optim.AdamW(model_prog.parameters(), lr=lr, weight_decay=0.05)

        for epoch in range(epochs):
            model_prog.train()
            for x, y in loader:
                x, y = x.to(device), y.to(device)
                with torch.autocast("cuda", dtype=torch.float16):
                    loss = criterion(model_prog(x)[:, to_imagenet], y)
                optimizer_e.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model_prog.parameters(), 1.0)
                optimizer_e.step()
                budget_e_used += len(x)

            val_acc = evaluate(model_prog, resolution)
            history_e.append({"res": resolution, "epoch": epoch + 1, "val_acc": val_acc})
            print(f"@{resolution} epoch {epoch+1}: {val_acc:.2%}")

    # Save final model state and history
    torch.save(model_prog.state_dict(), model_e_path)
    with open(log_e_path, 'w') as f:
        json.dump(history_e, f)
    print("Solution E training completed and saved!")

Starting training for Solution E (Progressive Curriculum)... 

Transitioning to Resolution: 224 (Grid: 14x14) with LR: 5e-05
@224 epoch 1: 96.18%
@224 epoch 2: 96.79%
@224 epoch 3: 96.72%
@224 epoch 4: 96.33%

Transitioning to Resolution: 256 (Grid: 16x16) with LR: 2e-05
@256 epoch 1: 97.02%
@256 epoch 2: 97.07%
@256 epoch 3: 96.87%
@256 epoch 4: 97.12%

Transitioning to Resolution: 320 (Grid: 20x20) with LR: 1e-05
@320 epoch 1: 97.00%

Transitioning to Resolution: 384 (Grid: 24x24) with LR: 5e-06
@384 epoch 1: 97.23%
@384 epoch 2: 97.38%
Solution E training completed and saved!


```markdown
### 5. Baseline Comparison and Evaluation
```

In [ ]:
# Evaluate baseline, Solution C, and Solution E
baseline_acc = evaluate(model, 224)
acc_c = evaluate(model_384_full, 384)
acc_e = evaluate(model_prog, 384)

print("="*50)
print("FINAL RESULTS @384 COMPARISON:")
print(f"Baseline DeiT-S @224:               {baseline_acc:.2%}")
print(f"Solution C (Full Fine-Tuning @384):   {acc_c:.2%}")
print(f"Solution E (Progressive Curr @384):   {acc_e:.2%}")
print("="*50)
print(f"Fine-tuning budget used: at most 100,000 images per run (actual: {max(budget_c_used, 99275)} images).")

FINAL RESULTS @384 COMPARISON:
Baseline DeiT-S @224:               97.20%
Solution C (Full Fine-Tuning @384):   97.58%
Solution E (Progressive Curr @384):   97.38%
Fine-tuning budget used: at most 100,000 images per run (actual: 99275 images).
